# 01 — Data Preparation

Prepares the Harumanis mango dataset for YOLOv8 training.

**Steps:** explore raw images → upload to Roboflow → export YOLOv8 format → verify structure → check class balance

## 1. Install dependencies

In [ ]:
!pip install roboflow ultralytics opencv-python matplotlib pyyaml

## 2. Explore raw dataset

In [ ]:
import os, cv2, matplotlib.pyplot as plt

RAW_DIR = "../datasets/raw"
image_files = [f for f in os.listdir(RAW_DIR) if f.lower().endswith((".jpg",".jpeg",".png"))]
print(f"Total images: {len(image_files)}")

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, fname in zip(axes.flatten(), image_files[:9]):
    img = cv2.cvtColor(cv2.imread(os.path.join(RAW_DIR, fname)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(fname, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Upload to Roboflow for labeling

**Instructions:**
1. Create free account at roboflow.com
2. New project → Object Detection
3. Class labels: , , , 
4. Upload images, draw bounding boxes, assign stage class per mango
5. Run export cell below when done labeling

In [ ]:
from roboflow import Roboflow

RF_API_KEY   = "YOUR_ROBOFLOW_API_KEY"
RF_WORKSPACE = "your-workspace"
RF_PROJECT   = "mlharum-mango"
RF_VERSION   = 1

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download("yolov8", location="../datasets/labeled")
print(f"Dataset downloaded to: {dataset.location}")

## 4. Verify dataset structure

Expected layout after export:


In [ ]:
import yaml

LABELED_DIR = "../datasets/labeled"
with open(os.path.join(LABELED_DIR, "data.yaml")) as f:
    cfg = yaml.safe_load(f)

print("Classes:", cfg["names"])
print("Num classes:", cfg["nc"])

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(LABELED_DIR, split, "images")
    lbl_dir = os.path.join(LABELED_DIR, split, "labels")
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"{split:8s}: {n_img} images  {n_lbl} labels")

## 5. Check class distribution

In [ ]:
from collections import Counter

class_names = cfg["names"]
counter = Counter()

for split in ["train","valid","test"]:
    lbl_dir = os.path.join(LABELED_DIR, split, "labels")
    if not os.path.exists(lbl_dir): continue
    for lbl_file in os.listdir(lbl_dir):
        with open(os.path.join(lbl_dir, lbl_file)) as f:
            for line in f:
                counter[class_names[int(line.split()[0])]] += 1

print("Annotations per class:")
for cls, count in sorted(counter.items()):
    print(f"  {cls:15s}: {count:4d}  {chr(9608)*int(count//5)}")

if counter and min(counter.values()) < 50:
    print("
⚠️  Some classes have < 50 annotations. Collect more images for those stages.")

## 6. Visualise sample annotations

In [ ]:
COLORS = {"early":(0,200,0),"mid":(255,165,0),"late":(255,100,0),"preharvest":(220,0,0)}

def draw_boxes(img_path, lbl_path, names):
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    with open(lbl_path) as f:
        for line in f:
            cls_id, cx, cy, bw, bh = map(float, line.split())
            cls_id = int(cls_id)
            x1,y1 = int((cx-bw/2)*w), int((cy-bh/2)*h)
            x2,y2 = int((cx+bw/2)*w), int((cy+bh/2)*h)
            name = names[cls_id]
            color = COLORS.get(name,(200,200,200))
            cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
            cv2.putText(img,name,(x1,y1-6),cv2.FONT_HERSHEY_SIMPLEX,0.5,color,1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

train_imgs = os.listdir(os.path.join(LABELED_DIR,"train","images"))[:4]
fig, axes = plt.subplots(1, len(train_imgs), figsize=(20,5))
for ax, fname in zip(axes, train_imgs):
    stem = os.path.splitext(fname)[0]
    ip = os.path.join(LABELED_DIR,"train","images",fname)
    lp = os.path.join(LABELED_DIR,"train","labels",stem+".txt")
    if os.path.exists(lp):
        ax.imshow(draw_boxes(ip, lp, class_names))
    ax.axis("off")
plt.suptitle("Sample annotations")
plt.tight_layout()
plt.show()